In [1]:
import sys
import os
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt


PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.hospital_env import HospitalEnv


SEED = 42
random.seed(SEED)
np.random.seed(SEED)



def generate_synthetic_data(num_patients, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    DAY_MINUTES = 8 * 60

    arrival_time = np.random.randint(0, DAY_MINUTES, num_patients)

    
    slot_time = []
    for a in arrival_time:
        delay = np.random.randint(0, 120)
        slot = min(a + delay, DAY_MINUTES - 1)
        slot_time.append(slot)
    slot_time = np.array(slot_time)

    # Priority
    priority = np.random.choice(
        [0, 1],
        size=num_patients,
        p=[0.75, 0.25]
    )

    
    no_show_prob = []
    for p in priority:
        if p == 1:
            prob = np.random.uniform(0.02, 0.15)
        else:
            prob = np.random.uniform(0.1, 0.4)
        no_show_prob.append(prob)
    no_show_prob = np.array(no_show_prob)

    show = (np.random.rand(num_patients) > no_show_prob).astype(int)

    return pd.DataFrame({
        "patient_id": range(1, num_patients + 1),
        "arrival_time": arrival_time,
        "slot_time": slot_time,
        "priority": priority,
        "no_show_prob": no_show_prob,
        "show": show
    })



arrival_bins = np.linspace(0, 480, 6)
slot_bins = np.linspace(0, 480, 6)
no_show_bins = np.linspace(0, 1, 6)
icu_bins = np.linspace(0, 1, 6)

def discretize_state(state):
    arrival, slot, priority, no_show, icu_ratio = state

    return (
        np.digitize(arrival, arrival_bins),
        np.digitize(slot, slot_bins),
        int(priority),
        np.digitize(no_show, no_show_bins),
        np.digitize(icu_ratio, icu_bins)
    )



actions = [0, 1]

def get_Q(Q, state):
    if state not in Q:
        Q[state] = {a: 0.0 for a in actions}
    return Q[state]



alpha = 0.1
gamma = 0.9
epsilon_start = 1.0
epsilon_min = 0.05
epsilon_decay = 0.98
episodes = 30



dataset_sizes = [500, 2000, 5000]
scaling_results = {}



for size in dataset_sizes:

    print(f"\n==============================")
    print(f"Training on dataset size: {size}")
    print(f"==============================")

    data = generate_synthetic_data(size, seed=SEED)
    Q = {}
    epsilon = epsilon_start
   
    for ep in range(episodes):

        env = HospitalEnv(data)
        state = discretize_state(env.reset())
        done = False

        while not done:
            
            if np.random.rand() < epsilon:
                action = np.random.choice(actions)
            else:
                action = max(get_Q(Q, state), key=get_Q(Q, state).get)

            next_state, reward, done, info = env.step(action)

            if not done:
                next_state = discretize_state(next_state)
                best_next = max(get_Q(Q, next_state).values())
            else:
                best_next = 0

            get_Q(Q, state)[action] += alpha * (
                reward + gamma * best_next - get_Q(Q, state)[action]
            )

            state = next_state
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    
    env = HospitalEnv(data)
    state = discretize_state(env.reset())
    done = False
    total_reward = 0
    steps = 0

    while not done:
        action = max(get_Q(Q, state), key=get_Q(Q, state).get)
        next_state, reward, done, info = env.step(action)

        total_reward += reward
        steps += 1

        if not done:
            state = discretize_state(next_state)

    avg_reward = total_reward / steps
    scaling_results[size] = avg_reward

    print(f"Average Reward per Patient (size={size}): {avg_reward:.3f}")



plt.figure()
plt.plot(
    list(scaling_results.keys()),
    list(scaling_results.values()),
    marker="o"
)
plt.axhline(0, linestyle="--")
plt.xlabel("Number of Patients")
plt.ylabel("Average Reward per Patient")
plt.title("Phase 5: Q-Learning Scalability Analysis")
plt.grid(True)
plt.savefig("scaling_analysis.png")
plt.close()



scaling_df = pd.DataFrame({
    "dataset_size": list(scaling_results.keys()),
    "avg_reward": list(scaling_results.values())
})

scaling_df.to_csv("scaling_results.csv", index=False)



print("\nFinal Scaling Results:")
for k, v in scaling_results.items():
    print(f"Patients: {k} -> Avg Reward: {v:.3f}")

print("\nFiles saved:")
print("scaling_analysis.png")
print("scaling_results.csv")




Training on dataset size: 500
Average Reward per Patient (size=500): -0.222

Training on dataset size: 2000
Average Reward per Patient (size=2000): -0.099

Training on dataset size: 5000
Average Reward per Patient (size=5000): -0.236

Final Scaling Results:
Patients: 500 -> Avg Reward: -0.222
Patients: 2000 -> Avg Reward: -0.099
Patients: 5000 -> Avg Reward: -0.236

Files saved:
scaling_analysis.png
scaling_results.csv
